# PyDI Data Integration Workflow: Papers

This notebook consolidates the paper use case into one executable workflow: source-schema mapping, blocking, entity matching, and data fusion.

## Table of Contents

1. Schema matching and source-schema translation
   - Schema matching evaluation
2. Blocking
3. Entity matching
4. Data fusion

## Part 1: Schema Matching

In [ ]:
import time 
start_time = time.time()

from pathlib import Path
import json
import logging
import math
import re
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# Resolve paths whether the notebook is run from the repo root or from usecases/papers.
BASE_DIR = Path.cwd()
if not (BASE_DIR / "input" / "data").exists():
    candidate = BASE_DIR / "usecases" / "papers"
    if candidate.exists():
        BASE_DIR = candidate.resolve()

REPO_ROOT = BASE_DIR.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from PyDI.schemamatching import SchemaMappingEvaluator

DATA_DIR = BASE_DIR / "input" / "data"
ENTITYMATCHING_INPUT_DIR = BASE_DIR / "input" / "entitymatching"
FUSION_INPUT_DIR = BASE_DIR / "input" / "fusion"
OUTPUT_DIR = BASE_DIR / "output"
SCHEMA_OUTPUT_DIR = OUTPUT_DIR / "schema_matching"
EM_OUTPUT_DIR = OUTPUT_DIR / "entity_matching"
FUSION_OUTPUT_DIR = OUTPUT_DIR / "data_fusion"

for path in [OUTPUT_DIR, SCHEMA_OUTPUT_DIR, EM_OUTPUT_DIR, FUSION_OUTPUT_DIR, OUTPUT_DIR / "logs"]:
    path.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)-5s] %(name)s - %(message)s",
    handlers=[logging.FileHandler(OUTPUT_DIR / "logs" / "papers_workflow.log"), logging.StreamHandler()],
    force=True,
)

print("Base directory:", BASE_DIR)
print("Input data:", DATA_DIR)
print("Entity matching input:", ENTITYMATCHING_INPUT_DIR)
print("Fusion input:", FUSION_INPUT_DIR)


In [ ]:
# Source files expected by the original schema-mapping cells.
# The *_FILTERED_PATH names are kept from the original schema-mapping code;
# they point to the task's source files in input/data/.
OPENALEX_FILTERED_PATH = DATA_DIR / "open_alex.jsonl"
DBLP_FILTERED_PATH = DATA_DIR / "dblp.jsonl"
CROSSREF_FILTERED_PATH = DATA_DIR / "crossref.jsonl"

for name, path in {
    "open_alex_filtered": OPENALEX_FILTERED_PATH,
    "dblp_filtered": DBLP_FILTERED_PATH,
    "crossref_filtered": CROSSREF_FILTERED_PATH,
}.items():
    print(f"{name}: {path}")
    if path.exists():
        preview = pd.read_json(path, lines=True, nrows=2)
        print(f"  columns: {list(preview.columns)}")

In [ ]:

TARGET_SCHEMA_PATH = BASE_DIR / "input" / "schemamatching" / "target_schema.json"
with TARGET_SCHEMA_PATH.open("r", encoding="utf-8") as handle:
    target_schema = json.load(handle)

TARGET_SCHEMA = [
    column
    for column in target_schema["properties"].keys()
    if column != "id"
]

schema_correspondences = pd.DataFrame(
    [
        # OpenAlex
        ("open_alex", "work_kind", "target", "type"),
        ("open_alex", "display_title", "target", "title"),
        ("open_alex", "authors_list", "target", "authors"),
        ("open_alex", "year_published", "target", "publication_year"),
        ("open_alex", "source_name", "target", "journal"),
        ("open_alex", "volume_tag", "target", "volume"),
        ("open_alex", "issue_tag", "target", "issue"),
        ("open_alex", "start_page", "target", "first_page"),
        ("open_alex", "end_page", "target", "last_page"),
        ("open_alex", "refs_count", "target", "referenced_works_count"),
        ("open_alex", "citations_count", "target", "cited_by_count"),
        # DBLP
        ("dblp", "entry_type", "target", "type"),
        ("dblp", "publication_title", "target", "title"),
        ("dblp", "author_list", "target", "authors"),
        ("dblp", "pub_year", "target", "publication_year"),
        ("dblp", "venue_name", "target", "journal"),
        ("dblp", "volume_no", "target", "volume"),
        ("dblp", "issue_no", "target", "issue"),
        ("dblp", "page_start", "target", "first_page"),
        ("dblp", "page_finish", "target", "last_page"),
        # Crossref
        ("crossref", "work_type", "target", "type"),
        ("crossref", "title_text", "target", "title"),
        ("crossref", "contributor_names", "target", "authors"),
        ("crossref", "issued_year", "target", "publication_year"),
        ("crossref", "container_title", "target", "journal"),
        ("crossref", "volume_id", "target", "volume"),
        ("crossref", "issue_id", "target", "issue"),
        ("crossref", "page_first", "target", "first_page"),
        ("crossref", "page_last", "target", "last_page"),
        ("crossref", "reference_total", "target", "referenced_works_count"),
        ("crossref", "cited_total", "target", "cited_by_count"),
    ],
    columns=["source_dataset", "source_column", "target_dataset", "target_column"],
)

print("Loaded target schema:", TARGET_SCHEMA_PATH)
print("Target fields:", ["id", *TARGET_SCHEMA])
display(schema_correspondences)


## Schema Matching Evaluation


In [ ]:
# Evaluate the hand-written paper schema correspondences against the curated gold mapping.
with (BASE_DIR / "input" / "schemamatching" / "sm_mapping_gold.json").open("r", encoding="utf-8") as handle:
    schema_mapping_gold = json.load(handle)

schema_mapping_gold_df = pd.DataFrame(schema_mapping_gold["mappings"])
schema_mapping_predictions = schema_correspondences.copy()
schema_mapping_predictions["target_dataset"] = schema_mapping_predictions["target_dataset"].replace({
    "target": "target_schema",
})
schema_mapping_predictions["score"] = 1.0

schema_matching_metrics = SchemaMappingEvaluator.evaluate(
    schema_mapping_predictions,
    schema_mapping_gold_df,
    complete=True,
)

schema_matching_summary = pd.DataFrame([
    {
        **schema_matching_metrics,
        "predicted_mappings": len(schema_mapping_predictions),
        "gold_mappings": len(schema_mapping_gold_df),
    }
])

per_source_schema_matching = pd.DataFrame([
    {
        "source_dataset": source_dataset,
        **SchemaMappingEvaluator.evaluate(
            schema_mapping_predictions[
                schema_mapping_predictions["source_dataset"] == source_dataset
            ],
            schema_mapping_gold_df[
                schema_mapping_gold_df["source_dataset"] == source_dataset
            ],
            complete=True,
        ),
    }
    for source_dataset in sorted(schema_mapping_gold_df["source_dataset"].unique())
])

display(schema_matching_summary)
per_source_schema_matching


In [ ]:
def normalize_doi(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    value = str(value).strip().lower()
    for prefix in ("https://doi.org/", "http://doi.org/", "doi:"):
        if value.startswith(prefix):
            value = value[len(prefix):]
    return value.rstrip(".") or None


# Author-list normalization helper.
def normalize_authors_list(data):
    """
    Normalizes various author string/list formats into a clean list of names.
    Handles strings, lists, np.nan, AND np.ndarray.
    """
    if isinstance(data, (list, np.ndarray)):
        return [str(item).strip().lower() for item in data if str(item).strip()]
    if pd.isna(data):
        return []
    text = str(data).strip().lower().strip('[]')
    return [author.strip() for author in text.split(',') if author.strip()]

In [ ]:

from collections.abc import Mapping, Sequence
from PyDI.io import load_json

TARGET_COLUMNS = list(TARGET_SCHEMA)


def _already_target_shaped(df: pd.DataFrame) -> bool:
    return set(TARGET_COLUMNS).issubset(df.columns)


def _write_schema_output(df: pd.DataFrame, new_name: str):
    SCHEMA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = SCHEMA_OUTPUT_DIR / f"{new_name}.jsonl"
    out = df.copy()
    for col in TARGET_COLUMNS:
        if col not in out.columns:
            out[col] = np.nan
    out = out[TARGET_COLUMNS]
    out.to_json(output_path, orient="records", lines=True)
    print(f"stored in {output_path}")


def schemamapping(columns_new, df, new_name):
    available_cols = [c for c in columns_new.keys() if c in df.columns]
    df2 = df[available_cols].copy()
    df2.rename(columns=columns_new, inplace=True)
    _write_schema_output(df2, new_name)


def _safe_nested(obj, path, default=None):
    """Safely get nested keys: handles None/NaN/non-dict/JSON-strings."""
    if obj is None or (isinstance(obj, float) and pd.isna(obj)):
        return default
    if isinstance(obj, str):
        try:
            obj = json.loads(obj)
        except Exception:
            return default
    if not isinstance(obj, Mapping):
        return default
    cur = obj
    for k in path:
        if not isinstance(cur, Mapping):
            return default
        cur = cur.get(k)
        if cur is None:
            return default
    return cur


def extract_author_names(authorships):
    if isinstance(authorships, str):
        try:
            authorships = json.loads(authorships)
        except Exception:
            return None
    if not isinstance(authorships, Sequence):
        return None
    names = []
    for a in authorships:
        if isinstance(a, Mapping):
            name = _safe_nested(a, ["author", "display_name"])
            if name:
                names.append(name)
        elif isinstance(a, str) and a:
            names.append(a)
    return names if names else None


def map_openalex(path: Path):
    df = pd.read_json(path, lines=True)
    if _already_target_shaped(df):
        _write_schema_output(df, "open_alex")
        return

    if "authors_list" in df.columns:
        df.loc[:, "authors_list"] = df["authors_list"].map(extract_author_names)
    if "start_page" in df.columns:
        df["start_page"] = df["start_page"].astype("string").str.split(":").str[-1]
    if "end_page" in df.columns:
        df["end_page"] = df["end_page"].astype("string").str.split(":").str[-1]
        df["end_page"] = df["end_page"].fillna(df.get("start_page"))

    columns_new = {
        "work_kind": "type", "display_title": "title",
        "authors_list": "authors", "year_published": "publication_year",
        "source_name": "journal", "volume_tag": "volume",
        "issue_tag": "issue", "start_page": "first_page", "end_page": "last_page",
        "refs_count": "referenced_works_count", "citations_count": "cited_by_count",
    }
    schemamapping(columns_new, df, "open_alex")


def map_dblp(path: Path):
    df = pd.read_json(path, lines=True)
    if _already_target_shaped(df):
        _write_schema_output(df, "dblp")
        return

    columns_new = {
        "entry_type": "type", "publication_title": "title",
        "author_list": "authors", "pub_year": "publication_year", "venue_name": "journal",
        "volume_no": "volume", "issue_no": "issue", "page_start": "first_page",
        "page_finish": "last_page",
    }
    schemamapping(columns_new, df, "dblp")


def map_crossref(path: Path):
    df = pd.read_json(path, lines=True)
    if _already_target_shaped(df):
        _write_schema_output(df, "crossref")
        return

    columns_new = {
        "work_type": "type", "title_text": "title",
        "contributor_names": "authors", "issued_year": "publication_year",
        "container_title": "journal",
        "volume_id": "volume", "issue_id": "issue", "page_first": "first_page",
        "page_last": "last_page", "reference_total": "referenced_works_count",
        "cited_total": "cited_by_count",
    }
    schemamapping(columns_new, df, "crossref")


map_openalex(OPENALEX_FILTERED_PATH)
map_dblp(DBLP_FILTERED_PATH)
map_crossref(CROSSREF_FILTERED_PATH)


In [ ]:
# Dataset loading and author normalization,
# with paths adapted to the schema outputs produced above.
df_dblp = load_json(SCHEMA_OUTPUT_DIR / "dblp.jsonl", add_index=True, lines=True)
df_dblp.rename(columns={"dblp_id": "id"}, inplace=True)
df_crossref = load_json(SCHEMA_OUTPUT_DIR / "crossref.jsonl", add_index=True, lines=True)
df_crossref.rename(columns={"crossref_id": "id"}, inplace=True)
df_openalex = load_json(SCHEMA_OUTPUT_DIR / "open_alex.jsonl", add_index=True, lines=True)
df_openalex.rename(columns={"open_alex_id": "id"}, inplace=True)

df_dblp['authors_normalized'] = df_dblp['authors'].apply(normalize_authors_list)
df_crossref['authors_normalized'] = df_crossref['authors'].apply(normalize_authors_list)
df_openalex['authors_normalized'] = df_openalex['authors'].apply(normalize_authors_list)

DATASETS = {
    "dblp": df_dblp,
    "crossref": df_crossref,
    "open_alex": df_openalex,
}

for name, df in DATASETS.items():
    df.attrs["dataset_name"] = name
    print(f"{name}: {len(df):,} rows")
    display(df.head(2))

In [ ]:
profile_rows = []
for name, df in DATASETS.items():
    for col in TARGET_SCHEMA:
        profile_rows.append({
            "dataset": name,
            "column": col,
            "non_null": int(df[col].notna().sum()),
            "density": float(df[col].notna().mean()),
        })
profile = pd.DataFrame(profile_rows)
display(profile.pivot(index="column", columns="dataset", values="density").round(3))

## Part 2: Blocking

In [ ]:
from PyDI.entitymatching.blocking import TokenBlocker
from PyDI.entitymatching.evaluation import EntityMatchingEvaluator
import ast
import re

SPLIT_DIR = ENTITYMATCHING_INPUT_DIR
MATCH_OUTPUT_DIR = EM_OUTPUT_DIR / "matches"
BLOCKING_OUTPUT_DIR = EM_OUTPUT_DIR / "blocking" / "token_blocker_author_ngram2"
BLOCKING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Selected blocker:
# TokenBlocker(column='authors', ngram_size=2, ngram_type='word', preprocess=clean_authors)
# Set this to False after a first run if you want to reuse this notebook's cached materialized candidates.
REBUILD_BLOCKING_CANDIDATES = True


def load_split(pair_name: str, split: str) -> pd.DataFrame:
    path = SPLIT_DIR / f"{pair_name}_{split}.csv"
    df = pd.read_csv(path)
    rename = {"id_dblp": "id1", "id_crossref": "id2", "id_openalex": "id2"}
    return df.rename(columns=rename)[["id1", "id2", "label"]]


def clean_authors(author_data):
    """Author cleaner of the selected blocker."""
    if not author_data:
        return ""

    name_list = []
    if isinstance(author_data, list):
        name_list = author_data
    elif isinstance(author_data, str):
        try:
            evaluated_data = ast.literal_eval(author_data)
            if isinstance(evaluated_data, list):
                name_list = evaluated_data
            else:
                name_list = [str(evaluated_data)]
        except (ValueError, SyntaxError):
            name_list = author_data.split(',')
    else:
        name_list = [str(author_data)]

    clean_names = [str(name).strip() for name in name_list if name and str(name).strip()]
    text = " ".join(clean_names)
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return " ".join(text.split())


def materialize_winner_blocker(pair_name: str, df_left: pd.DataFrame, df_right: pd.DataFrame) -> pd.DataFrame:
    cache_path = BLOCKING_OUTPUT_DIR / f"{pair_name}_author_token_blocking_candidates.csv"
    if cache_path.exists() and not REBUILD_BLOCKING_CANDIDATES:
        print(f"{pair_name}: loading cached candidates from {cache_path}")
        return pd.read_csv(cache_path)

    blocker_token = TokenBlocker(
        df_left,
        df_right,
        column='authors',
        output_dir=BLOCKING_OUTPUT_DIR / pair_name,
        id_column='id',
        ngram_size=2,
        ngram_type='word',
        preprocess=clean_authors,
    )
    candidates = blocker_token.materialize()
    candidates.to_csv(cache_path, index=False)
    print(f"{pair_name}: materialized {len(candidates):,} candidates with exact winner blocker")
    return candidates


candidates_crossref = materialize_winner_blocker("dblp_crossref", df_dblp, df_crossref)
candidates_openalex = materialize_winner_blocker("dblp_openalex", df_dblp, df_openalex)

df_train_crossref = load_split("dblp_crossref", "train")
df_train_openalex = load_split("dblp_openalex", "train")
df_validation_crossref = load_split("dblp_crossref", "val")
df_validation_openalex = load_split("dblp_openalex", "val")
df_test_crossref = load_split("dblp_crossref", "test")
df_test_openalex = load_split("dblp_openalex", "test")

# Backward-compatible names used by later summary cells.
test_crossref = df_test_crossref
test_openalex = df_test_openalex

display(candidates_crossref.head())


In [ ]:
blocking_summary = []
for pair_name, candidates, right_df, test_pairs in [
    ("dblp_crossref", candidates_crossref, df_crossref, test_crossref),
    ("dblp_openalex", candidates_openalex, df_openalex, test_openalex),
]:
    metrics = EntityMatchingEvaluator.evaluate_blocking(
        candidate_pairs=candidates[["id1", "id2"]],
        test_pairs=test_pairs,
        total_possible_pairs=len(df_dblp) * len(right_df),
        out_dir=EM_OUTPUT_DIR / "blocking" / pair_name,
    )
    blocking_summary.append({"pair": pair_name, **metrics})

blocking_summary = pd.DataFrame(blocking_summary)
display(blocking_summary[["pair", "total_candidates", "pair_completeness", "pair_quality", "reduction_ratio"]])

## Part 3: Entity Matching

In [ ]:
from PyDI.entitymatching.comparators import StringComparator, DateComparator
from PyDI.entitymatching import FeatureExtractor, MLBasedMatcher

extractor_refined = FeatureExtractor([
    StringComparator("title", "overlap", tokenization="word", preprocess=str.lower),
    StringComparator("title", "jaro_winkler", preprocess=str.lower),
    StringComparator("title", "ratcliff_obershelp", preprocess=str.lower),
    StringComparator("title", "mra", preprocess=str.lower),
    StringComparator("title", "prefix", preprocess=str.lower),
    StringComparator("title", "jaccard", tokenization="ngram_3", preprocess=str.lower),
    StringComparator(
        column="authors_normalized",
        similarity_function="jaccard",
        tokenization="word",
        list_strategy="concatenate",
    ),
])

feature_names = [
    "title_overlap_word",
    "title_jaro_winkler",
    "title_ratcliff_obershelp",
    "title_mra_phonetic",
    "title_prefix",
    "title_jaccard_ngram3",
    "authors_bow_jaccard_word",
]

print("Refined feature set used by the final XGBoost model:")
display(pd.DataFrame({"feature": feature_names}))

In [ ]:
RUN_FULL_XGBOOST_INFERENCE = True

if RUN_FULL_XGBOOST_INFERENCE:
    from xgboost import XGBClassifier

    train_crossref = df_train_crossref
    train_openalex = df_train_openalex

    features_crossref = extractor_refined.create_features(df_dblp, df_crossref, train_crossref, "id", train_crossref["label"])
    features_openalex = extractor_refined.create_features(df_dblp, df_openalex, train_openalex, "id", train_openalex["label"])

    xgb_crossref = XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    xgb_openalex = XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    xgb_crossref.fit(features_crossref.drop(["label", "id1", "id2"], axis=1), features_crossref["label"])
    xgb_openalex.fit(features_openalex.drop(["label", "id1", "id2"], axis=1), features_openalex["label"])

    matcher_crossref = MLBasedMatcher(extractor_refined)
    matcher_openalex = MLBasedMatcher(extractor_refined)
    matches_crossref = matcher_crossref.match(df_dblp, df_crossref, candidates_crossref, "id", xgb_crossref, threshold=0.5, use_probabilities=True)
    matches_openalex = matcher_openalex.match(df_dblp, df_openalex, candidates_openalex, "id", xgb_openalex, threshold=0.5, use_probabilities=True)

    matches_crossref.to_csv(MATCH_OUTPUT_DIR / "matches_dblp-crossref.csv", index=False)
    matches_openalex.to_csv(MATCH_OUTPUT_DIR / "matches_dblp-openalex.csv", index=False)
else:
    matches_crossref = pd.read_csv(MATCH_OUTPUT_DIR / "matches_dblp-crossref.csv")
    matches_openalex = pd.read_csv(MATCH_OUTPUT_DIR / "matches_dblp-openalex.csv")
    matches_crossref = matches_crossref[["id1", "id2", "score", "notes"]].copy()
    matches_openalex = matches_openalex[["id1", "id2", "score", "notes"]].copy()

print(f"DBLP-Crossref matches: {len(matches_crossref):,}")
print(f"DBLP-OpenAlex matches: {len(matches_openalex):,}")
display(matches_crossref.head())


In [ ]:
# Evaluation on the fixed validation splits from input/entitymatching.
xgb_eval_rows = []
for pair_name, matches, validation_pairs in [
    ("dblp_crossref", matches_crossref, df_validation_crossref),
    ("dblp_openalex", matches_openalex, df_validation_openalex),
]:
    metrics = EntityMatchingEvaluator.evaluate_matching(
        correspondences=matches,
        test_pairs=validation_pairs,
    )
    xgb_eval_rows.append({"pair": pair_name, "validation_set": "val", **metrics})

matching_summary = pd.DataFrame(xgb_eval_rows)
display(matching_summary[["pair", "validation_set", "precision", "recall", "f1", "accuracy", "filtered_correspondences"]])


In [ ]:
from PyDI.entitymatching import GreedyOneToOneMatchingAlgorithm

# Final declustering before test evaluation.
greedy_matching_crossref = GreedyOneToOneMatchingAlgorithm()
greedy_matches_crossref = greedy_matching_crossref.cluster(matches_crossref)

greedy_matching_openalex = GreedyOneToOneMatchingAlgorithm()
greedy_matches_openalex = greedy_matching_openalex.cluster(matches_openalex)

final_eval_crossref = EntityMatchingEvaluator.evaluate_matching(
    correspondences=greedy_matches_crossref,
    test_pairs=df_test_crossref,
)
final_eval_openalex = EntityMatchingEvaluator.evaluate_matching(
    correspondences=greedy_matches_openalex,
    test_pairs=df_test_openalex,
)

final_matching_summary = pd.DataFrame([
    {"pair": "dblp_crossref", **final_eval_crossref},
    {"pair": "dblp_openalex", **final_eval_openalex},
])
display(final_matching_summary[["pair", "precision", "recall", "f1", "accuracy", "filtered_correspondences"]])

# These are the exact final correspondence variables used downstream.
correspondences_crossref_final = greedy_matches_crossref
correspondences_openalex_final = greedy_matches_openalex


## Part 4: Data Fusion

In [ ]:
from PyDI.fusion import (
    DataFusionStrategy,
    DataFusionEngine,
    DataFusionEvaluator,
    longest_string,
    prefer_higher_trust,
    union,
    tokenized_match,
    year_only_match,
    most_recent,
    voting,
    maximum,
    intersection,
    weighted_voting,
    shortest_string,
)

# Recreate the shape produced by PyDI load_json(..., add_index=True, name=name)
# in the original fusion code: source-specific ID columns and no generic entity-matching id.
def fusion_input_from_mapped(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df.drop(columns=["id", "authors_normalized"], errors="ignore").copy()
    out.insert(0, f"{name}_id", df["id"].astype(str).to_numpy())
    out = out.drop(columns=["publisher"], errors="ignore")
    out.attrs["dataset_name"] = name
    return out

loaded = {
    "dblp": fusion_input_from_mapped(df_dblp, "dblp"),
    "crossref": fusion_input_from_mapped(df_crossref, "crossref"),
    "open_alex": fusion_input_from_mapped(df_openalex, "open_alex"),
}

match_files = {
    "dblp-crossref": correspondences_crossref_final,
    "dblp-openalex": correspondences_openalex_final,
}

filtered_loaded = loaded

df_test_set = pd.read_json(FUSION_INPUT_DIR / "fusion_test.jsonl", lines=True)

# Gold-standard dtype normalization.
for col in ["issue", "first_page", "last_page", "volume"]:
    if col in df_test_set.columns:
        ser = df_test_set[col]
        ser_num = pd.to_numeric(ser, errors="coerce")
        ser_int = ser_num.astype("Int64")
        ser_str = ser_int.astype("string")
        mask = ser_num.isna() & ser.notna()
        ser_str.loc[mask] = ser.loc[mask].astype(str)
        df_test_set[col] = ser_str

all_correspondences = pd.concat(
    [
        match_files["dblp-crossref"][["id1", "id2", "score"]],
        match_files["dblp-openalex"][["id1", "id2", "score"]],
    ],
    ignore_index=True,
)
print(f"Correspondences passed to fusion: {len(all_correspondences):,}")


In [ ]:
import ast
import re
from PyDI.fusion import FusionResult

# Helper functions and the V5 fusion setup.
def _normalize_pages(df):
    df = df.copy()
    for col in ["first_page", "last_page"]:
        if col not in df.columns:
            continue
        def _fix(val):
            if isinstance(val, str):
                s = val.strip()
                if re.fullmatch(r"\d+:\d+", s):
                    a, b = s.split(":")
                    nums = sorted([int(a), int(b)])
                    return str(nums[0]) if col == "first_page" else str(nums[1])
                if s.isdigit():
                    return s
            return val
        df[col] = df[col].apply(_fix)
    for col in ["first_page", "last_page"]:
        if col in df.columns:
            ser_num = pd.to_numeric(df[col], errors="coerce")
            ser_int = ser_num.astype("Int64")
            df[col] = ser_int.astype("string")
    return df


def normalize_authors_field(df):
    df = df.copy()
    if 'authors' not in df.columns:
        return df
    def _norm(val):
        if isinstance(val, list):
            parts = [str(v).strip() for v in val if str(v).strip()]
        elif isinstance(val, str):
            s = val.strip()
            try:
                if s.startswith('[') and s.endswith(']'):
                    parsed = ast.literal_eval(s)
                    if isinstance(parsed, list):
                        parts = [str(v).strip() for v in parsed if str(v).strip()]
                    else:
                        parts = []
                else:
                    parts = [p.strip() for p in s.split(',') if p.strip()]
            except Exception:
                parts = [p.strip() for p in s.split(',') if p.strip()]
        else:
            parts = []
        parts = [' '.join(p.split()) for p in parts if p]
        return parts
    df['authors'] = df['authors'].apply(_norm)
    return df


def normalize_journal_field(df):
    df = df.copy()
    if 'journal' not in df.columns:
        return df
    def _norm(val):
        if not isinstance(val, str):
            return val
        s = val.strip()
        if s.lower().startswith('the '):
            s = s[4:]
        s = s.replace('&amp;', '&')
        return s
    df['journal'] = df['journal'].apply(_norm)
    return df


def normalize_count_fields(df):
    df = df.copy()
    if 'referenced_works_count' in df.columns:
        ser = pd.to_numeric(df['referenced_works_count'], errors='coerce')
        ser = ser.where(ser != 0, pd.NA)
        df['referenced_works_count'] = ser
    if 'cited_by_count' in df.columns:
        df['cited_by_count'] = pd.to_numeric(df['cited_by_count'], errors='coerce')
    return df


def normalize_title_field(df):
    df = df.copy()
    if 'title' not in df.columns:
        return df
    def _norm(val):
        if not isinstance(val, str):
            return val
        return val.rstrip(' .')
    df['title'] = df['title'].apply(_norm)
    return df


normalized_loaded_v5 = {}
for name, df in filtered_loaded.items():
    trust_val = df.attrs.get('trust')
    norm_df = normalize_count_fields(
        normalize_journal_field(normalize_title_field(normalize_authors_field(_normalize_pages(df))))
    )
    if trust_val is not None:
        norm_df.attrs['trust'] = trust_val
    normalized_loaded_v5[name] = norm_df

for name, trust_val in [('dblp', 1), ('crossref', 3), ('open_alex', 1)]:
    if name in normalized_loaded_v5:
        normalized_loaded_v5[name].attrs['trust'] = trust_val

gold_norm_v5 = normalize_count_fields(
    normalize_journal_field(normalize_title_field(normalize_authors_field(_normalize_pages(df_test_set))))
)

strategy_v5 = DataFusionStrategy("Version_5_JournalClean")
strategy_v5.add_attribute_fuser("type", voting)
strategy_v5.add_attribute_fuser("title", longest_string)
strategy_v5.add_attribute_fuser("authors", prefer_higher_trust)
strategy_v5.add_attribute_fuser("publication_year", most_recent)
strategy_v5.add_attribute_fuser("journal", voting)
strategy_v5.add_attribute_fuser("volume", voting)
strategy_v5.add_attribute_fuser("issue", voting)
strategy_v5.add_attribute_fuser("first_page", voting)
strategy_v5.add_attribute_fuser("last_page", prefer_higher_trust, trust_map={'crossref': 3, 'open_alex': 1, 'dblp': 1})
strategy_v5.add_attribute_fuser("referenced_works_count", prefer_higher_trust, trust_map={'crossref': 3, 'open_alex': 1, 'dblp': 1})
strategy_v5.add_attribute_fuser("cited_by_count", prefer_higher_trust, trust_map={'crossref': 1, 'open_alex': 3, 'dblp': 1})

engine_v5 = DataFusionEngine(strategy_v5, debug=True, debug_file=FUSION_OUTPUT_DIR / "fusion_debug_v5.jsonl")
fused_v5 = engine_v5.run(
    datasets=list(normalized_loaded_v5.values()),
    correspondences=[
        match_files['dblp-crossref'][['id1', 'id2', 'score']],
        match_files['dblp-openalex'][['id1', 'id2', 'score']],
    ],
    id_column={'dblp': 'dblp_id', 'crossref': 'crossref_id', 'open_alex': 'open_alex_id'},
)

fused_v5["_id"] = fused_v5["_fusion_sources"].apply(lambda sources: "+".join(sorted(sources)))
fused_v5.to_json(FUSION_OUTPUT_DIR / "papers_fused_v5.jsonl", orient="records", lines=True)
print(f"Fused records: {len(fused_v5):,}")
print("Saved to", FUSION_OUTPUT_DIR / "papers_fused_v5.jsonl")
display(fused_v5.head())


In [ ]:
# --- MaDI-Bench normalization output: the normalized source tables, one CSV per source
# (record id + target-schema columns), the input of madi_bench.evaluation.score_normalization,
# which scores them against input/normalization/<split>.csv.
import json as _json
from pathlib import Path as _Path
_norm_dir = OUTPUT_DIR / "normalization"
_norm_dir.mkdir(parents=True, exist_ok=True)
_target = list(_json.loads((_Path(BASE_DIR) / "input" / "schemamatching" / "target_schema.json").read_text())["properties"])
for _name, _df in {_n: _df.rename(columns={f"{_n}_id": "id"}) for _n, _df in normalized_loaded_v5.items()}.items():
    _id_col = "id" if "id" in _df.columns else next((c for c in _df.columns if c.endswith("_id")), _df.columns[0])
    _cols = [_id_col] + [c for c in _df.columns if c in _target and c != "id" and c != _id_col]
    _df[_cols].rename(columns={_id_col: "id"}).to_csv(_norm_dir / f"{_name}.csv", index=False)
    print(f"normalized table {_name}: {len(_df)} rows, {len(_cols)} columns -> {_norm_dir / (_name + '.csv')}")


In [ ]:
# --- MaDI-Bench fusion evaluation: the public evaluation package scores the
# fused table on the validation and the test split -- strict comparators on the v2 gold set,
# all-gold accuracy (a gold record the pipeline did not produce counts as wrong) with its coverage.
# This is the scorer the benchmark's reference numbers were produced with; the notebook's own
# comparator registrations above are not used for scoring.
import json as _json, os as _os, sys as _sys
from pathlib import Path as _Path
try:
    from madi_bench.evaluation import score_fusion as _score_fusion
except ImportError:
    _sys.path.insert(0, _os.environ.get("MADI_BENCH_ROOT") or str(_Path(BASE_DIR).resolve().parents[2]))
    from madi_bench.evaluation import score_fusion as _score_fusion
_fused_out = fused_v5.copy()
# the notebook's own column names for target-schema attributes (the same aliases the benchmark's
# rescoring applies, reproduction/scoring/rescore.py): scoring is by target-schema name
for _src, _dst in ():
    if _src in _fused_out.columns and _dst not in _fused_out.columns:
        _fused_out = _fused_out.rename(columns={_src: _dst})
_fusion_dir = OUTPUT_DIR / "data_fusion"
_fusion_dir.mkdir(parents=True, exist_ok=True)
_fused_out.to_csv(_fusion_dir / "fused.csv", index=False)
_scores = {}
for _split in ("validation", "test"):
    _r = _score_fusion(BASE_DIR, _fused_out, split=_split)
    _scores[_split] = _r
    print(f"{_split}: all-gold accuracy {_r['overall_accuracy_all_gold']:.4f} "
          f"(evaluated {_r['overall_accuracy_evaluated']:.4f} on {_r['n_gold_evaluated']}/{_r['n_gold']} gold records, "
          f"attribute cell coverage {_r['attribute_cell_coverage']:.2f})")
    for _a, _v in sorted(_r["per_attribute_accuracy_all_gold"].items()):
        print(f"    {_a:28s} {_v:.4f}")
(_fusion_dir / "fusion_scores_v2.json").write_text(_json.dumps(_scores, indent=1, default=str))


## Outputs

The consolidated workflow writes schema-mapped intermediate files to `output/schema_matching`, entity-matching diagnostics to `output/entity_matching`, and the fused JSONL dataset plus debug logs to `output/data_fusion`.

In [ ]:
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nTotal execution time: {elapsed_time:.2f} seconds")